# Parametric study: mechanism × chemistry model × KODES balancer

One tutorial case, run over the full grid of

1. **mechanisms** — every `0.<mechanism>` the case has (`h2o2`, `grimech`, `C1_C3_HT_NOX_159_2459`);
2. **chemistry models** — `standard` (OpenFOAM's, on the CPU) and `kodes` (`kodesChemistryModel`, on the GPU);
3. **KODES balancers** — `none`, `temperature`, `rhsNorm`, `stiffness`.

Every point of the grid is a [foamlib](https://github.com/gerlero/foamlib) clone of the tutorial in
`runs/<case>/<mechanism>.<variant>`, set up and run by [kodesCase.py](kodesCase.py) — the same code
the cases' `./Allrun` uses, so a run here is the run `./Allrun <mechanism> <variant>` would make.
The tutorial itself is never touched.

The `kodes` points need a CUDA device and `libkodes_<mechanism>.so` (the top-level `./Allwmake`);
without them they fail at start-up and show up below as failed, with the reason. A point that
already finished is not rerun unless `RERUN = True`, so the notebook can be re-executed after
adding a mechanism or a balancer and only the new points run.

Run it from the repository's `.venv` (see the top-level README), with OpenFOAM sourced.

In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from foamlib import CalledProcessError, FoamCase

import kodesCase as kc

## Parameters

`CONTROL` is applied to each clone's `system/controlDict` before it runs — the cases' own `endTime`
is a demonstration length, which for the 159-specie mechanism on the CPU is hours; shorten it here
for a first pass. `FIELDS` are compared against the `standard` run of the same mechanism.

In [ ]:
CASE = "nonUniformHeating2D"

MECHANISMS = None                     # None: every mechanism the case has
CHEMISTRY = ["standard", "kodes"]
BALANCERS = ["none", "temperature", "rhsNorm", "stiffness"]

CONTROL = {}                          # e.g. {"endTime": 1e-4, "writeInterval": 1e-4}
NPROCS = 1                            # MPI ranks per run
FIELDS = ("T",)                       # compared against standard at the last common time

RERUN = False
STUDY = kc.TUTORIALS.parent / "runs" / CASE

## The grid

`standard` has no balancer, so it is one point per mechanism; `kodes` is one per balancer.

In [ ]:
tutorial = FoamCase(kc.TUTORIALS / CASE)
mechanisms = MECHANISMS or kc.mechanisms(tutorial)

rows = []
for mechanism in mechanisms:
    for chemistry in CHEMISTRY:
        for balancer in ([None] if chemistry == "standard" else BALANCERS):
            variant = "standard" if balancer is None else f"kodes-{balancer}"
            rows.append(dict(mechanism=mechanism, chemistry=chemistry,
                             balancer=balancer or "-", variant=variant))

grid = pd.DataFrame(rows)
grid["path"] = [STUDY / f"{r.mechanism}.{r.variant}" for r in grid.itertuples()]
grid

## Running

Points run one after another: the `kodes` ones share the one GPU, and a `standard` one timed
alongside another would not be timing the integrator alone.

In [ ]:
def finished(path, mechanism, variant):
    if not path.is_dir():
        return False
    log = kc.solver_log(path, mechanism, variant)
    return log.is_file() and kc.read_log(log)["completed"]


def run_point(row):
    if not RERUN and finished(row.path, row.mechanism, row.variant):
        return "done before"

    shutil.rmtree(row.path, ignore_errors=True)
    case = kc.clone(tutorial, row.path)

    control = case.control_dict
    with control:
        for key, value in CONTROL.items():
            control[key] = value

    try:
        kc.run(case, row.mechanism, row.variant, nprocs=NPROCS)
    except CalledProcessError:
        return "failed"
    return "ran"


STUDY.mkdir(parents=True, exist_ok=True)
for row in grid.itertuples():
    print(f"{row.mechanism:24s} {row.variant:18s}", end=" ", flush=True)
    print(run_point(row))

## Results

From each run's solver log: whether it reached `End`, how many steps it took, its wall clock, and
the wall clock per step with the first step left out — start-up (reading the mechanism, opening the
GPU) lands in that one and says nothing about the integrator. For the `kodes` runs the model's banner
is read too, to confirm which balancer actually ran.

In [ ]:
def read_point(row):
    log = kc.solver_log(row.path, row.mechanism, row.variant)
    if not log.is_file():
        return {"completed": False, "error": "no solver log"}
    summary = kc.read_log(log)
    kodes = summary.pop("kodes")
    summary["balancing"] = kodes.get("Batch balancing")
    summary["concurrent"] = kodes.get("Concurrent systems")
    return summary


results = pd.concat(
    [grid, pd.DataFrame([read_point(r) for r in grid.itertuples()])], axis=1
)
results[["mechanism", "variant", "completed", "steps", "time", "clockTime",
         "stepTime", "balancing", "error"]]

### Speed-up and agreement with `standard`

Speed-up is `standard`'s wall clock per step over the variant's, for the same mechanism. The field
differences are taken at the last time both runs wrote. For `C1_C3_HT_NOX_159_2459` they are not a
like-for-like comparison: the `standard` path drops the pressure dependence of its 236 PLOG reactions
(see `tutorials/mechanisms/C1_C3_HT_NOX_159_2459/README.md`).

In [ ]:
reference = results[results.variant == "standard"].set_index("mechanism")

def against_standard(row):
    out = {"speedup": np.nan}
    if not row.completed or row.mechanism not in reference.index:
        return out
    ref = reference.loc[row.mechanism]
    if ref.completed and row.stepTime:
        out["speedup"] = ref.stepTime / row.stepTime
    if ref.completed and row.variant != "standard":
        try:
            out.update(kc.compare(row.path, ref.path, FIELDS))
        except (ValueError, KeyError) as e:
            out["compareError"] = str(e)
    return out


results = pd.concat(
    [results, pd.DataFrame([against_standard(r) for r in results.itertuples()])], axis=1
)
results.drop(columns="path").to_csv(STUDY / "results.csv", index=False)

columns = ["mechanism", "variant", "stepTime", "speedup"] + [
    c for c in results.columns if c.endswith((".maxAbs", ".relL2"))
]
results.loc[results.completed, columns]

## Plots

Wall clock per step for every point, on a log scale once the mechanisms span more than an
order of magnitude; then the speed-up of each balancer over `standard`; then each balancer against
`kodes-none`, which is what the balancing alone is worth. Missing bars are points that did not
complete — the results table above says why.

In [ ]:
# Categorical slots in fixed order: one colour per variant, the same in every plot
COLORS = dict(zip(
    ["standard", "kodes-none", "kodes-temperature", "kodes-rhsNorm", "kodes-stiffness"],
    ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"],
))
INK, MUTED, GRID = "#1f1f1e", "#6b6a64", "#e4e3dd"

plt.rcParams.update({
    "axes.edgecolor": GRID, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})


def grouped_bars(ax, table, variants, ylabel, log=False):
    """One group per mechanism, one bar per variant; NaN leaves a gap."""
    width = 0.8 / len(variants)
    x = np.arange(len(table.index))
    for i, variant in enumerate(variants):
        values = table[variant] if variant in table else pd.Series(np.nan, table.index)
        ax.bar(x + (i - (len(variants) - 1) / 2) * width, values,
               width * 0.92, color=COLORS[variant], label=variant)
    ax.set_xticks(x, table.index)
    ax.set_ylabel(ylabel)
    # A log scale only once the bars span more than a decade: below that it
    # just cuts the bars off above zero
    span = table[[v for v in variants if v in table]].stack()
    if log and len(span) and span.max() > 10 * span.min():
        ax.set_yscale("log")
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.grid(axis="x", visible=False)
    ax.legend(loc="upper left", ncols=len(variants), fontsize="small",
              labelcolor=INK, bbox_to_anchor=(0, 1.12))


done = results[results.completed]
variants = [v for v in COLORS if v in set(grid.variant)]
kodes_variants = [v for v in variants if v != "standard"]

if done.empty:
    print("Nothing completed yet.")
else:
    step = done.pivot(index="mechanism", columns="variant", values="stepTime")
    step = step.reindex([m for m in mechanisms if m in step.index])

    fig, ax = plt.subplots(figsize=(9, 4))
    grouped_bars(ax, step, variants, "wall clock per step [s]", log=True)
    ax.set_title(f"{CASE}: cost per time step", loc="left", pad=28)
    plt.show()

In [ ]:
speedup = done.pivot(index="mechanism", columns="variant", values="speedup")

if done.empty or not set(kodes_variants) & set(speedup.columns):
    print("No completed kodes run to compare with standard.")
else:
    speedup = speedup.reindex([m for m in mechanisms if m in speedup.index])
    fig, ax = plt.subplots(figsize=(9, 4))
    grouped_bars(ax, speedup, kodes_variants, "speed-up over standard")
    ax.axhline(1, color=MUTED, linewidth=1)
    ax.set_title(f"{CASE}: GPU over CPU, per balancer", loc="left", pad=28)
    plt.show()

In [ ]:
balanced = [v for v in kodes_variants if v != "kodes-none"]

if done.empty or "kodes-none" not in set(done.variant) or not balanced:
    print("Needs a completed kodes-none run and at least one balanced one.")
else:
    step = done.pivot(index="mechanism", columns="variant", values="stepTime")
    gain = step[[v for v in balanced if v in step]].rdiv(step["kodes-none"], axis=0)
    gain = gain.reindex([m for m in mechanisms if m in gain.index])

    fig, ax = plt.subplots(figsize=(9, 4))
    grouped_bars(ax, gain, balanced, "speed-up over kodes-none")
    ax.axhline(1, color=MUTED, linewidth=1)
    ax.set_title(f"{CASE}: what batch balancing is worth", loc="left", pad=28)
    plt.show()

## One run in detail

Every point is an ordinary OpenFOAM case, so foamlib reads its fields directly — here the
temperature at each written time of one point, against its `standard` reference.

In [ ]:
POINT = ("h2o2", "kodes-temperature")   # (mechanism, variant)

path = STUDY / f"{POINT[0]}.{POINT[1]}"
ref_path = STUDY / f"{POINT[0]}.standard"

def temperature_history(path):
    case = FoamCase(path)
    return pd.DataFrame(
        [(t.time, np.min(t["T"].internal_field), np.max(t["T"].internal_field))
         for t in case if "T" in t],
        columns=["time", "Tmin", "Tmax"],
    )

fig, ax = plt.subplots(figsize=(9, 4))
for p, label, color in [(ref_path, "standard", COLORS["standard"]),
                        (path, POINT[1], COLORS.get(POINT[1], INK))]:
    if p.is_dir() and len(FoamCase(p)) > 1:
        h = temperature_history(p)
        ax.plot(h.time * 1e3, h.Tmax, color=color, linewidth=2, marker="o",
                markersize=4, label=f"{label}, max")
        ax.plot(h.time * 1e3, h.Tmin, color=color, linewidth=2, linestyle="--",
                label=f"{label}, min")
ax.set_xlabel("time [ms]")
ax.set_ylabel("T [K]")
ax.set_title(f"{CASE}, {POINT[0]}: temperature range", loc="left")
if ax.lines:
    ax.legend(fontsize="small", labelcolor=INK)
plt.show()